# QuantRisk AI

## Module 01 — Data Collection and Preprocessing

### Objective

The objective of this notebook is to load, understand, validate,
clean and organize historical NIFTY 50 stock market data before
performing exploratory data analysis and feature engineering.

The dataset contains historical OHLCV data for NIFTY 50 companies,
along with NIFTY 50 Index and India VIX data.

### Data Period
2020-01-01 to 2025-12-31

### Main Tasks

1. Load historical stock datasets
2. Understand dataset structure
3. Validate column names and data types
4. Convert dates into proper datetime format
5. Check missing values
6. Check duplicate observations
7. Validate OHLC prices
8. Inspect trading volume
9. Validate all NIFTY 50 datasets
10. Prepare clean data for further analysis

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
file_path = "../research_project/RELIANCE.csv"

reliance = pd.read_csv(
    file_path,
    skiprows=[1, 2]
)

reliance.head()

,Price,Close,High,Low,Open,Volume
0,2020-01-01,672.216125,680.008790,670.390471,675.956609,14004468
1,2020-01-02,683.660278,686.176208,673.284921,673.284921,17710316
2,2020-01-03,684.484009,686.487835,678.183067,682.636001,20984698
3,2020-01-06,668.609253,680.364982,667.050708,676.847198,24519177
4,2020-01-07,678.895630,683.304036,673.952826,676.401934,16683622


In [3]:
reliance.tail()

,Price,Close,High,Low,Open,Volume
1482,2025-12-24,1551.028687,1568.448147,1546.449881,1565.461954,8815745
1483,2025-12-26,1552.024048,1553.815812,1547.146696,1547.544758,2311495
1484,2025-12-29,1538.486694,1550.929166,1536.495899,1547.743942,5972105
1485,2025-12-30,1532.713379,1546.449794,1530.722584,1539.880194,8815884
1486,2025-12-31,1563.172607,1569.742208,1533.907890,1533.907890,5771830


In [4]:
reliance.shape

(1487, 6)

In [5]:
reliance.rename(
    columns={"Price": "Date"},
    inplace=True
)

In [6]:
reliance.head()

,Date,Close,High,Low,Open,Volume
0,2020-01-01,672.216125,680.008790,670.390471,675.956609,14004468
1,2020-01-02,683.660278,686.176208,673.284921,673.284921,17710316
2,2020-01-03,684.484009,686.487835,678.183067,682.636001,20984698
3,2020-01-06,668.609253,680.364982,667.050708,676.847198,24519177
4,2020-01-07,678.895630,683.304036,673.952826,676.401934,16683622


In [7]:
reliance.columns

Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')

In [8]:
reliance["Date"] = pd.to_datetime(reliance["Date"])

In [9]:
reliance.dtypes

Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
dtype: object

In [10]:
reliance.isnull().sum()

Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [11]:
reliance.duplicated().sum()

np.int64(0)

In [12]:
print("Start Date:", reliance["Date"].min())
print("End Date:", reliance["Date"].max())

Start Date: 2020-01-01 00:00:00
End Date: 2025-12-31 00:00:00


In [13]:
# ==========================================
# OHLC Financial Data Validation
# ==========================================

invalid_ohlc = reliance[
    (reliance["High"] < reliance["Open"]) |
    (reliance["High"] < reliance["Close"]) |
    (reliance["Low"] > reliance["Open"]) |
    (reliance["Low"] > reliance["Close"])
]

print("Invalid OHLC rows:", len(invalid_ohlc))

Invalid OHLC rows: 0


In [14]:
# Check for zero or negative prices

price_columns = ["Open", "High", "Low", "Close"]

invalid_prices = (reliance[price_columns] <= 0).sum()

print(invalid_prices)

Open     0
High     0
Low      0
Close    0
dtype: int64


In [15]:
# Check trading volume

print("Negative volume rows:", (reliance["Volume"] < 0).sum())
print("Zero volume rows:", (reliance["Volume"] == 0).sum())

Negative volume rows: 0
Zero volume rows: 1


In [16]:
zero_volume_row = reliance[reliance["Volume"] == 0]

zero_volume_row

,Date,Close,High,Low,Open,Volume
1291,2025-03-18,1228.242798,1228.242798,1228.242798,1228.242798,0


### Data Quality Observation

One zero-volume observation was identified for Reliance Industries on
18 March 2025. On this date, Open, High, Low, and Close prices are identical.

The observation is retained temporarily rather than removed or imputed.
A consistent treatment strategy will be decided after examining zero-volume
observations across all NIFTY 50 stocks.

In [17]:
print("Dates sorted:", reliance["Date"].is_monotonic_increasing)

Dates sorted: True


# Tata Motors is excluded temporarily because the historical
# TATAMOTORS symbol changed following the 2025 demerger.

### Tata Motors Corporate Action

Tata Motors is temporarily excluded from the automated dataset pipeline.

The company underwent a demerger in October 2025, after which the
historical TATAMOTORS symbol changed. Therefore, the available file
does not contain valid historical observations.

Rather than artificially combining data across the corporate action,
Tata Motors is treated as a special case and excluded from the initial
modeling dataset.

In [18]:
def load_stock_data(file_path):
    
    # Read the CSV-formatted stock file
    df = pd.read_csv(
        file_path,
        skiprows=[1, 2]
    )
    
    # Rename the first column from Price to Date
    if "Price" in df.columns:
        df.rename(columns={"Price": "Date"}, inplace=True)
    
    # Convert Date column to datetime
    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )
    
    # Sort observations chronologically
    df = df.sort_values("Date")
    
    # Reset row numbers
    df.reset_index(drop=True, inplace=True)
    
    return df

In [19]:
reliance_test = load_stock_data(
    "../research_project/RELIANCE.csv"
)

reliance_test.head()

,Date,Close,High,Low,Open,Volume
0,2020-01-01,672.216125,680.008790,670.390471,675.956609,14004468
1,2020-01-02,683.660278,686.176208,673.284921,673.284921,17710316
2,2020-01-03,684.484009,686.487835,678.183067,682.636001,20984698
3,2020-01-06,668.609253,680.364982,667.050708,676.847198,24519177
4,2020-01-07,678.895630,683.304036,673.952826,676.401934,16683622


In [20]:
reliance_test.shape

(1487, 6)

In [21]:
reliance_test.dtypes

Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
dtype: object

In [22]:
def validate_stock_data(df, stock_name):
    
    # Required columns
    required_columns = [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]
    
    # Check whether required columns exist
    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]
    
    # Total missing values
    missing_values = df.isnull().sum().sum()
    
    # Duplicate rows
    duplicate_rows = df.duplicated().sum()
    
    # Invalid OHLC relationships
    invalid_ohlc = (
        (df["High"] < df["Open"]) |
        (df["High"] < df["Close"]) |
        (df["Low"] > df["Open"]) |
        (df["Low"] > df["Close"])
    ).sum()
    
    # Zero or negative prices
    price_columns = [
        "Open",
        "High",
        "Low",
        "Close"
    ]
    
    invalid_prices = (
        df[price_columns] <= 0
    ).sum().sum()
    
    # Volume checks
    negative_volume = (
        df["Volume"] < 0
    ).sum()
    
    zero_volume = (
        df["Volume"] == 0
    ).sum()
    
    # Check chronological order
    dates_sorted = (
        df["Date"].is_monotonic_increasing
    )
    
    # Create validation result
    result = {
        "Stock": stock_name,
        "Rows": len(df),
        "Missing_Values": missing_values,
        "Duplicate_Rows": duplicate_rows,
        "Missing_Columns": len(missing_columns),
        "Invalid_OHLC": invalid_ohlc,
        "Invalid_Prices": invalid_prices,
        "Negative_Volume": negative_volume,
        "Zero_Volume": zero_volume,
        "Dates_Sorted": dates_sorted,
        "Start_Date": df["Date"].min(),
        "End_Date": df["Date"].max()
    }
    
    return result

In [23]:
reliance_report = validate_stock_data(
    reliance_test,
    "RELIANCE"
)

reliance_report

{'Stock': 'RELIANCE',
 'Rows': 1487,
 'Missing_Values': np.int64(0),
 'Duplicate_Rows': np.int64(0),
 'Missing_Columns': 0,
 'Invalid_OHLC': np.int64(0),
 'Invalid_Prices': np.int64(0),
 'Negative_Volume': np.int64(0),
 'Zero_Volume': np.int64(1),
 'Dates_Sorted': True,
 'Start_Date': Timestamp('2020-01-01 00:00:00'),
 'End_Date': Timestamp('2025-12-31 00:00:00')}

In [24]:

dataset_path = Path("../research_project/")

all_files = list(dataset_path.glob("*.csv"))

print("Total .csv files found:", len(all_files))

for file in all_files:
    print(file.name)

Total .csv files found: 51
ADANIENT.csv
ADANIPORTS.csv
APOLLOHOSP.csv
ASIANPAINT.csv
AXISBANK.csv
BAJAJ-AUTO.csv
BAJAJFINSV.csv
BAJFINANCE.csv
BEL.csv
BHARTIARTL.csv
CIPLA.csv
COALINDIA.csv
DRREDDY.csv
EICHERMOT.csv
ETERNAL.csv
GRASIM.csv
HCLTECH.csv
HDFCBANK.csv
HDFCLIFE.csv
HEROMOTOCO.csv
HINDALCO.csv
HINDUNILVR.csv
ICICIBANK.csv
IndiaVIX.csv
INDUSINDBK.csv
INFY.csv
ITC.csv
JIOFIN.csv
JSWSTEEL.csv
KOTAKBANK.csv
LT.csv
M&M.csv
MARUTI.csv
NESTLEIND.csv
NIFTY50.csv
NTPC.csv
ONGC.csv
POWERGRID.csv
RELIANCE.csv
SBILIFE.csv
SBIN.csv
SHRIRAMFIN.csv
SUNPHARMA.csv
TATACONSUM.csv
TATASTEEL.csv
TCS.csv
TECHM.csv
TITAN.csv
TRENT.csv
ULTRACEMCO.csv
WIPRO.csv


In [25]:
# ==========================================
# Separate Stock and Market Datasets
# ==========================================

market_files = ["NIFTY50.csv", "IndiaVIX.csv"]

stock_files = [
    file for file in all_files
    if file.name not in market_files
]

print("Total files:", len(all_files))
print("Stock datasets:", len(stock_files))
print("Market datasets:", len(market_files))

Total files: 51
Stock datasets: 49
Market datasets: 2


In [ ]:
# ==========================================
# Validate All Stock Datasets
# ==========================================

validation_results = []

for file in stock_files:
    
    stock_name = file.stem
    
    try:
        # Load stock dataset
        stock_df = load_stock_data(file)
        
        # Validate stock dataset
        report = validate_stock_data(
            stock_df,
            stock_name
        )
        
        # Store validation result
        validation_results.append(report)
        
        print(f"{stock_name}: Successfully validated")
        
    except Exception as error:  # noqa: BLE001
        
        print(
            f"{stock_name}: Validation failed -> {error}"
        )

ADANIENT: Successfully validated
ADANIPORTS: Successfully validated
APOLLOHOSP: Successfully validated
ASIANPAINT: Successfully validated
AXISBANK: Successfully validated
BAJAJ-AUTO: Successfully validated
BAJAJFINSV: Successfully validated
BAJFINANCE: Successfully validated
BEL: Successfully validated
BHARTIARTL: Successfully validated
CIPLA: Successfully validated
COALINDIA: Successfully validated
DRREDDY: Successfully validated
EICHERMOT: Successfully validated
ETERNAL: Successfully validated
GRASIM: Successfully validated
HCLTECH: Successfully validated
HDFCBANK: Successfully validated
HDFCLIFE: Successfully validated
HEROMOTOCO: Successfully validated
HINDALCO: Successfully validated
HINDUNILVR: Successfully validated
ICICIBANK: Successfully validated
INDUSINDBK: Successfully validated
INFY: Successfully validated
ITC: Successfully validated
JIOFIN: Successfully validated
JSWSTEEL: Successfully validated
KOTAKBANK: Successfully validated
LT: Successfully validated
M&M: Successfull

In [27]:
# Convert validation results into DataFrame

quality_report = pd.DataFrame(
    validation_results
)

quality_report

,Stock,Rows,Missing_Values,Duplicate_Rows,Missing_Columns,Invalid_OHLC,Invalid_Prices,Negative_Volume,Zero_Volume,Dates_Sorted,Start_Date,End_Date
0,ADANIENT,1487,0,0,0,0,0,0,1,True,2020-01-01,2025-12-31
1,ADANIPORTS,1487,0,0,0,0,0,0,1,True,2020-01-01,2025-12-31
2,APOLLOHOSP,1487,0,0,0,0,0,0,1,True,2020-01-01,2025-12-31
3,ASIANPAINT,1487,0,0,0,0,0,0,1,True,2020-01-01,2025-12-31
4,AXISBANK,1487,0,0,0,0,0,0,1,True,2020-01-01,2025-12-31
5,BAJAJ-AUTO,1487,0,0,0,0,0,0,2,True,2020-01-01,2025-12-31
6,BAJAJFINSV,1487,0,0,0,0,0,0,1,True,2020-01-01,2025-12-31
7,BAJFINANCE,1487,0,0,0,0,0,0,1,True,2020-01-01,2025-12-31
8,BEL,1487,0,0,0,0,0,0,1,True,2020-01-01,2025-12-31
9,BHARTIARTL,1487,0,0,0,0,0,0,1,True,2020-01-01,2025-12-31


In [28]:
# ==========================================
# Investigate Zero-Volume Observations
# ==========================================

zero_volume_records = []

for file in stock_files:
    
    stock_name = file.stem
    df = load_stock_data(file)
    
    zero_rows = df[df["Volume"] == 0].copy()
    
    if not zero_rows.empty:
        zero_rows["Stock"] = stock_name
        zero_volume_records.append(zero_rows)


zero_volume_df = pd.concat(
    zero_volume_records,
    ignore_index=True
)

zero_volume_df[
    ["Stock", "Date", "Open", "High", "Low", "Close", "Volume"]
]

,Stock,Date,Open,High,Low,Close,Volume
0,ADANIENT,2025-03-18,2250.577393,2250.577393,2250.577393,2250.577393,0
1,ADANIPORTS,2025-03-18,1126.895142,1126.895142,1126.895142,1126.895142,0
2,APOLLOHOSP,2025-03-18,6140.627441,6140.627441,6140.627441,6140.627441,0
3,ASIANPAINT,2025-03-18,2176.945068,2176.945068,2176.945068,2176.945068,0
4,AXISBANK,2025-03-18,1032.470703,1032.470703,1032.470703,1032.470703,0
5,BAJAJ-AUTO,2024-01-15,6966.018066,6966.018066,6966.018066,6966.018066,0
6,BAJAJ-AUTO,2025-03-18,7199.689941,7199.689941,7199.689941,7199.689941,0
7,BAJAJFINSV,2025-03-18,1869.093628,1869.093628,1869.093628,1869.093628,0
8,BAJFINANCE,2025-03-18,847.562744,847.562744,847.562744,847.562744,0
9,BEL,2025-03-18,278.098175,278.098175,278.098175,278.098175,0


In [29]:
zero_volume_df["Date"].value_counts().sort_index()

Date
2021-11-04     1
2022-09-12     1
2022-10-24     1
2024-01-15     4
2025-03-18    48
2025-04-11     1
2025-04-15     1
2025-09-08     1
Name: count, dtype: int64

### Investigation of Zero-Volume Observations

Zero-volume observations were examined across all stock datasets.

A strong systematic pattern was identified on 18 March 2025, where
48 of the 49 stock datasets contained zero trading volume.

Because the issue occurs across almost the entire stock universe on the
same date, it is treated as a potential source-data anomaly rather than
an individual stock-specific error.

Several additional isolated zero-volume observations were also identified.

At this stage, these observations are retained rather than deleted or
imputed. Their treatment will be reconsidered during feature engineering,
particularly for volume-based indicators.

In [30]:
quality_report[
    quality_report["Rows"] != 1487
][
    ["Stock", "Rows", "Start_Date", "End_Date"]
]

,Stock,Rows,Start_Date,End_Date
14,ETERNAL,1099,2021-07-23,2025-12-31
26,JIOFIN,584,2023-08-21,2025-12-31


In [31]:
quality_report.to_csv(
    "../notebooks/stock_data_quality_report.csv",
    index=False
)

print("Data quality report saved successfully.")

Data quality report saved successfully.


In [66]:
problem_date = pd.Timestamp("2025-03-18")

stocks_with_zero_volume = set(
    zero_volume_df.loc[
        zero_volume_df["Date"] == problem_date,
        "Stock"
    ]
)

all_stock_names = {
    file.stem for file in stock_files
}

stocks_without_zero_volume = (
    all_stock_names - stocks_with_zero_volume
)

print(
    "Stock(s) without zero volume on 2025-03-18:",
    stocks_without_zero_volume
)

Stock(s) without zero volume on 2025-03-18: {'ITC'}


In [33]:
# ==========================================
# Load NIFTY 50 Index Dataset
# ==========================================

nifty_file = "../research_project/NIFTY50.csv"

nifty = pd.read_csv(
    nifty_file,
    skiprows=[1, 2]
)

nifty.head()

,Price,Close,High,Low,Open,Volume
0,2020-01-01,12182.500000,12222.200195,12165.299805,12202.150391,304100
1,2020-01-02,12282.200195,12289.900391,12195.250000,12198.549805,407700
2,2020-01-03,12226.650391,12265.599609,12191.349609,12261.099609,428800
3,2020-01-06,11993.049805,12179.099609,11974.200195,12170.599609,396500
4,2020-01-07,12052.950195,12152.150391,12005.349609,12079.099609,447800


In [34]:
print("Shape:", nifty.shape)

print("\nColumns:")
print(nifty.columns)

print("\nData Types:")
print(nifty.dtypes)

print("\nMissing Values:")
print(nifty.isnull().sum())

Shape: (1486, 6)

Columns:
Index(['Price', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')

Data Types:
Price      object
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

Missing Values:
Price     0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


In [35]:
# ==========================================
# Preprocess NIFTY 50 Index
# ==========================================

nifty.rename(
    columns={"Price": "Date"},
    inplace=True
)

nifty["Date"] = pd.to_datetime(
    nifty["Date"],
    errors="coerce"
)

nifty = nifty.sort_values("Date").reset_index(drop=True)

In [36]:
print("NIFTY:")
print(nifty.dtypes)


NIFTY:
Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
dtype: object


In [37]:
# ==========================================
# Load India VIX Dataset
# ==========================================

vix_file = "../research_project/IndiaVIX.csv"

vix = pd.read_csv(
    vix_file,
    skiprows=[1, 2]
)

vix.head()

,Price,Close,High,Low,Open,Volume
0,2020-01-02,11.49,11.700000,10.73,11.60,0
1,2020-01-03,12.70,12.940000,10.69,11.49,0
2,2020-01-06,14.78,15.100000,12.70,12.70,0
3,2020-01-07,14.61,14.910000,14.46,14.78,0
4,2020-01-08,15.64,16.389999,13.79,14.61,0


In [38]:
print("Shape:", vix.shape)

print("\nColumns:")
print(vix.columns)

print("\nData Types:")
print(vix.dtypes)

print("\nMissing Values:")
print(vix.isnull().sum())

Shape: (1470, 6)

Columns:
Index(['Price', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')

Data Types:
Price      object
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

Missing Values:
Price     0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


In [39]:
# ==========================================
# Preprocess India VIX
# ==========================================

vix.rename(
    columns={"Price": "Date"},
    inplace=True
)

vix["Date"] = pd.to_datetime(
    vix["Date"],
    errors="coerce"
)

vix = vix.sort_values("Date").reset_index(drop=True)

In [40]:
print("India Vix:")
print(vix.dtypes)


India Vix:
Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
dtype: object


In [41]:
nifty_report = validate_stock_data(
    nifty,
    "NIFTY50"
)

vix_report = validate_stock_data(
    vix,
    "IndiaVIX"
)

market_quality_report = pd.DataFrame([
    nifty_report,
    vix_report
])

market_quality_report

,Stock,Rows,Missing_Values,Duplicate_Rows,Missing_Columns,Invalid_OHLC,Invalid_Prices,Negative_Volume,Zero_Volume,Dates_Sorted,Start_Date,End_Date
0,NIFTY50,1486,0,0,0,0,0,0,14,True,2020-01-01,2025-12-31
1,IndiaVIX,1470,0,0,0,0,0,0,1470,True,2020-01-02,2025-12-31


In [42]:
print("NIFTY Start Date:", nifty["Date"].min())
print("NIFTY End Date:", nifty["Date"].max())

print()

print("VIX Start Date:", vix["Date"].min())
print("VIX End Date:", vix["Date"].max())

NIFTY Start Date: 2020-01-01 00:00:00
NIFTY End Date: 2025-12-31 00:00:00

VIX Start Date: 2020-01-02 00:00:00
VIX End Date: 2025-12-31 00:00:00


In [43]:
# Dates present in NIFTY but missing from VIX

nifty_dates = set(nifty["Date"])
vix_dates = set(vix["Date"])

missing_in_vix = sorted(
    nifty_dates - vix_dates
)

print(
    "Dates present in NIFTY but missing in VIX:",
    len(missing_in_vix)
)

missing_in_vix

Dates present in NIFTY but missing in VIX: 16


[Timestamp('2020-01-01 00:00:00'),
 Timestamp('2021-01-01 00:00:00'),
 Timestamp('2021-01-22 00:00:00'),
 Timestamp('2021-02-15 00:00:00'),
 Timestamp('2021-02-17 00:00:00'),
 Timestamp('2021-05-05 00:00:00'),
 Timestamp('2021-05-11 00:00:00'),
 Timestamp('2021-05-20 00:00:00'),
 Timestamp('2021-05-25 00:00:00'),
 Timestamp('2021-05-31 00:00:00'),
 Timestamp('2021-07-05 00:00:00'),
 Timestamp('2022-12-26 00:00:00'),
 Timestamp('2024-01-01 00:00:00'),
 Timestamp('2024-02-19 00:00:00'),
 Timestamp('2025-01-01 00:00:00'),
 Timestamp('2025-02-01 00:00:00')]

In [44]:
# Dates present in VIX but missing from NIFTY

missing_in_nifty = sorted(
    vix_dates - nifty_dates
)

print(
    "Dates present in VIX but missing in NIFTY:",
    len(missing_in_nifty)
)

missing_in_nifty

Dates present in VIX but missing in NIFTY: 0


[]

In [45]:
# ==========================================
# Select Required Market Variables
# ==========================================

nifty_market = nifty[
    ["Date", "Close"]
].copy()

vix_market = vix[
    ["Date", "Close"]
].copy()

In [46]:
nifty_market.rename(
    columns={"Close": "NIFTY_Close"},
    inplace=True
)

vix_market.rename(
    columns={"Close": "VIX_Close"},
    inplace=True
)

In [47]:
nifty_market.head()

,Date,NIFTY_Close
0,2020-01-01,12182.500000
1,2020-01-02,12282.200195
2,2020-01-03,12226.650391
3,2020-01-06,11993.049805
4,2020-01-07,12052.950195


In [48]:
vix_market.head()

,Date,VIX_Close
0,2020-01-02,11.49
1,2020-01-03,12.70
2,2020-01-06,14.78
3,2020-01-07,14.61
4,2020-01-08,15.64


In [49]:
# ==========================================
# Merge NIFTY 50 and India VIX
# ==========================================

market_data = pd.merge(
    nifty_market,
    vix_market,
    on="Date",
    how="left"
)

market_data.head()

,Date,NIFTY_Close,VIX_Close
0,2020-01-01,12182.500000,NaN
1,2020-01-02,12282.200195,11.49
2,2020-01-03,12226.650391,12.70
3,2020-01-06,11993.049805,14.78
4,2020-01-07,12052.950195,14.61


In [50]:
print("Market dataset shape:", market_data.shape)

print("\nMissing values:")
print(market_data.isnull().sum())

print("\nDuplicate dates:")
print(market_data["Date"].duplicated().sum())

print("\nDate range:")
print(
    market_data["Date"].min(),
    "to",
    market_data["Date"].max()
)

Market dataset shape: (1486, 3)

Missing values:
Date            0
NIFTY_Close     0
VIX_Close      16
dtype: int64

Duplicate dates:
0

Date range:
2020-01-01 00:00:00 to 2025-12-31 00:00:00


In [51]:
missing_vix_dates = market_data[
    market_data["VIX_Close"].isnull()
]

missing_vix_dates

,Date,NIFTY_Close,VIX_Close
0,2020-01-01,12182.500000,NaN
250,2021-01-01,14018.500000,NaN
265,2021-01-22,14371.900391,NaN
280,2021-02-15,15314.700195,NaN
282,2021-02-17,15208.900391,NaN
332,2021-05-05,14617.849609,NaN
336,2021-05-11,14850.750000,NaN
342,2021-05-20,14906.049805,NaN
345,2021-05-25,15208.450195,NaN
349,2021-05-31,15582.799805,NaN


### Market Data Alignment

NIFTY 50 and India VIX datasets were aligned using the Date column.

The NIFTY 50 dataset contains 1,486 observations, while India VIX
contains 1,470 observations.

After performing a left join using the NIFTY 50 trading calendar,
16 observations contain missing VIX values.

These missing values are retained at this stage rather than being
replaced with zero or artificially interpolated. Their treatment will
be determined during feature engineering based on the requirements
of the machine-learning models.

In [52]:
# ==========================================
# Create Master Stock Dataset
# ==========================================

stock_dataframes = []

for file in stock_files:
    
    stock_name = file.stem
    
    # Load stock using our reusable function
    df = load_stock_data(file)
    
    # Add stock identifier
    df["Stock"] = stock_name
    
    # Arrange columns
    df = df[
        [
            "Date",
            "Stock",
            "Open",
            "High",
            "Low",
            "Close",
            "Volume"
        ]
    ]
    
    stock_dataframes.append(df)


# Combine all stocks vertically
master_stocks = pd.concat(
    stock_dataframes,
    ignore_index=True
)

master_stocks.head(20)

,Date,Stock,Open,High,Low,Close,Volume
0,2020-01-01,ADANIENT,206.933131,208.368789,204.606365,205.794510,1553127
1,2020-01-02,ADANIENT,205.943023,211.091595,205.447967,209.111374,2991937
2,2020-01-03,ADANIENT,208.170764,210.250002,203.764775,206.240051,2512421
3,2020-01-06,ADANIENT,205.695488,205.695488,195.794382,197.576584,4353179
4,2020-01-07,ADANIENT,198.566702,203.665766,198.566702,202.032089,2966120
5,2020-01-08,ADANIENT,199.012223,201.537008,192.626013,199.507278,5762654
6,2020-01-09,ADANIENT,199.903342,206.834117,199.903342,205.943024,3496063
7,2020-01-10,ADANIENT,205.943022,210.893576,203.863784,207.032150,2769132
8,2020-01-13,ADANIENT,208.566787,214.111412,207.527183,211.487625,3527267
9,2020-01-14,ADANIENT,211.339131,213.368861,209.754963,212.230240,1775182


In [53]:
print("Master stock dataset shape:", master_stocks.shape)

print(
    "Number of unique stocks:",
    master_stocks["Stock"].nunique()
)

print(
    "Start Date:",
    master_stocks["Date"].min()
)

print(
    "End Date:",
    master_stocks["Date"].max()
)

print("\nMissing Values:")
print(master_stocks.isnull().sum())

print(
    "\nDuplicate Stock-Date combinations:",
    master_stocks.duplicated(
        subset=["Date", "Stock"]
    ).sum()
)

Master stock dataset shape: (71572, 7)
Number of unique stocks: 49
Start Date: 2020-01-01 00:00:00
End Date: 2025-12-31 00:00:00

Missing Values:
Date      0
Stock     0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

Duplicate Stock-Date combinations: 0


In [54]:
expected_rows = quality_report["Rows"].sum()

actual_rows = len(master_stocks)

print("Expected rows:", expected_rows)
print("Actual rows:", actual_rows)

print(
    "Row count validation:",
    expected_rows == actual_rows
)

Expected rows: 71572
Actual rows: 71572
Row count validation: True


In [55]:
# ==========================================
# Merge Stock Data with Market Data
# ==========================================

master_data = pd.merge(
    master_stocks,
    market_data,
    on="Date",
    how="left"
)

master_data.head()

,Date,Stock,Open,High,Low,Close,Volume,NIFTY_Close,VIX_Close
0,2020-01-01,ADANIENT,206.933131,208.368789,204.606365,205.794510,1553127,12182.500000,NaN
1,2020-01-02,ADANIENT,205.943023,211.091595,205.447967,209.111374,2991937,12282.200195,11.49
2,2020-01-03,ADANIENT,208.170764,210.250002,203.764775,206.240051,2512421,12226.650391,12.70
3,2020-01-06,ADANIENT,205.695488,205.695488,195.794382,197.576584,4353179,11993.049805,14.78
4,2020-01-07,ADANIENT,198.566702,203.665766,198.566702,202.032089,2966120,12052.950195,14.61


In [56]:
print("Final dataset shape:", master_data.shape)

print(
    "Unique stocks:",
    master_data["Stock"].nunique()
)

print("\nMissing Values:")
print(master_data.isnull().sum())

print(
    "\nDuplicate Stock-Date rows:",
    master_data.duplicated(
        subset=["Date", "Stock"]
    ).sum()
)

Final dataset shape: (71572, 9)
Unique stocks: 49

Missing Values:
Date             0
Stock            0
Open             0
High             0
Low              0
Close            0
Volume           0
NIFTY_Close     47
VIX_Close      808
dtype: int64

Duplicate Stock-Date rows: 0


In [57]:
# ==========================================
# Investigate Missing NIFTY Values
# ==========================================

missing_nifty = master_data[
    master_data["NIFTY_Close"].isnull()
]

print("Missing NIFTY rows:", len(missing_nifty))

print("\nMissing NIFTY dates:")
print(
    missing_nifty["Date"]
    .value_counts()
    .sort_index()
)

Missing NIFTY rows: 47

Missing NIFTY dates:
Date
2020-11-14    47
Name: count, dtype: int64


In [58]:
# ==========================================
# Investigate Missing VIX Values
# ==========================================

missing_vix = master_data[
    master_data["VIX_Close"].isnull()
]

print("Missing VIX rows:", len(missing_vix))

print("\nMissing VIX dates:")
print(
    missing_vix["Date"]
    .value_counts()
    .sort_index()
)

Missing VIX rows: 808

Missing VIX dates:
Date
2020-01-01    47
2020-11-14    47
2021-01-01    47
2021-01-22    47
2021-02-15    47
2021-02-17    47
2021-05-05    47
2021-05-11    47
2021-05-20    47
2021-05-25    47
2021-05-31    47
2021-07-05    47
2022-12-26    48
2024-01-01    49
2024-02-19    49
2025-01-01    49
2025-02-01    49
Name: count, dtype: int64


In [59]:
missing_both = master_data[
    master_data["NIFTY_Close"].isnull()
    & master_data["VIX_Close"].isnull()
]

print("Rows missing both:", len(missing_both))

print("\nDates missing both:")
print(
    missing_both["Date"]
    .value_counts()
    .sort_index()
)

Rows missing both: 47

Dates missing both:
Date
2020-11-14    47
Name: count, dtype: int64


In [60]:
print(
    missing_vix["Date"]
    .value_counts()
    .sort_index()
)

Date
2020-01-01    47
2020-11-14    47
2021-01-01    47
2021-01-22    47
2021-02-15    47
2021-02-17    47
2021-05-05    47
2021-05-11    47
2021-05-20    47
2021-05-25    47
2021-05-31    47
2021-07-05    47
2022-12-26    48
2024-01-01    49
2024-02-19    49
2025-01-01    49
2025-02-01    49
Name: count, dtype: int64


In [61]:
# ==========================================
# Remove Rows with Missing Market Data
# ==========================================

master_clean = master_data.dropna(
    subset=["NIFTY_Close", "VIX_Close"]
).copy()

master_clean.reset_index(
    drop=True,
    inplace=True
)

In [62]:
print("Before cleaning:", master_data.shape)
print("After cleaning:", master_clean.shape)

print("\nMissing Values:")
print(master_clean.isnull().sum())

print(
    "\nUnique stocks:",
    master_clean["Stock"].nunique()
)

print(
    "\nDuplicate Stock-Date rows:",
    master_clean.duplicated(
        subset=["Date", "Stock"]
    ).sum()
)

Before cleaning: (71572, 9)
After cleaning: (70764, 9)

Missing Values:
Date           0
Stock          0
Open           0
High           0
Low            0
Close          0
Volume         0
NIFTY_Close    0
VIX_Close      0
dtype: int64

Unique stocks: 49

Duplicate Stock-Date rows: 0


In [63]:
master_clean.groupby("Stock").size().sort_values()

Stock
JIOFIN         580
ETERNAL       1094
ADANIENT      1470
JSWSTEEL      1470
KOTAKBANK     1470
LT            1470
M&M           1470
MARUTI        1470
NESTLEIND     1470
NTPC          1470
ONGC          1470
POWERGRID     1470
RELIANCE      1470
SBIN          1470
SHRIRAMFIN    1470
SUNPHARMA     1470
TATACONSUM    1470
TATASTEEL     1470
TCS           1470
TECHM         1470
TITAN         1470
TRENT         1470
SBILIFE       1470
ITC           1470
INFY          1470
INDUSINDBK    1470
ADANIPORTS    1470
APOLLOHOSP    1470
ASIANPAINT    1470
AXISBANK      1470
BAJAJ-AUTO    1470
BAJAJFINSV    1470
BAJFINANCE    1470
BEL           1470
BHARTIARTL    1470
CIPLA         1470
COALINDIA     1470
DRREDDY       1470
EICHERMOT     1470
GRASIM        1470
HCLTECH       1470
HDFCBANK      1470
HDFCLIFE      1470
HEROMOTOCO    1470
HINDALCO      1470
HINDUNILVR    1470
ICICIBANK     1470
ULTRACEMCO    1470
WIPRO         1470
dtype: int64

In [64]:
master_clean = master_clean.sort_values(
    by=["Date", "Stock"]
).reset_index(drop=True)

master_clean.head(10)

,Date,Stock,Open,High,Low,Close,Volume,NIFTY_Close,VIX_Close
0,2020-01-02,ADANIENT,205.943023,211.091595,205.447967,209.111374,2991937,12282.200195,11.49
1,2020-01-02,ADANIPORTS,361.349430,367.802952,359.963106,366.321014,5919760,12282.200195,11.49
2,2020-01-02,APOLLOHOSP,1404.730250,1484.607020,1404.730250,1470.294189,2000143,12282.200195,11.49
3,2020-01-02,ASIANPAINT,1688.672265,1697.063792,1681.506510,1688.342285,411266,12282.200195,11.49
4,2020-01-02,AXISBANK,746.305900,755.261570,743.917696,753.221680,5156046,12282.200195,11.49
5,2020-01-02,BAJAJ-AUTO,2603.730025,2618.213801,2568.325418,2575.711670,452413,12282.200195,11.49
6,2020-01-02,BAJAJFINSV,934.422608,949.611891,934.417622,947.086182,1578910,12282.200195,11.49
7,2020-01-02,BAJFINANCE,411.850740,417.266007,411.365067,412.438416,8638440,12282.200195,11.49
8,2020-01-02,BEL,29.862069,30.796648,29.639549,30.603798,26262750,12282.200195,11.49
9,2020-01-02,BHARTIARTL,428.686066,433.689454,428.119659,429.724518,4933053,12282.200195,11.49


In [65]:
# ==========================================
# Save Clean Master Dataset
# ==========================================

output_path = "../notebooks/master_dataset.csv"

master_clean.to_csv(
    output_path,
    index=False
)

print("Master dataset saved successfully.")
print("Location:", output_path)

Master dataset saved successfully.
Location: ../notebooks/master_dataset.csv


### Missing Market Data Treatment

After combining individual stock data with NIFTY 50 and India VIX,
missing market-level observations were identified.

The missing values occurred on a small number of trading dates and were
repeated across multiple stocks because the dataset is stored in long format.

Since NIFTY 50 and India VIX will be used as market-context variables in
subsequent modeling, observations without the required market information
were removed rather than replacing them with zero or artificially generated
values.

The original raw datasets remain unchanged. Cleaning was performed only on
the processed master dataset.